In [ ]:
import os, asyncio, httpx
import pandas as pd
from binance import AsyncClient, BinanceSocketManager
from dotenv import load_dotenv


load_dotenv('../../.env')
BINANCE_KEY = os.getenv('BINANCE_KEY')
BINANCE_SECRET = os.getenv('BINANCE_SECRET')

client = await AsyncClient.create(BINANCE_KEY, BINANCE_SECRET)

In [30]:


# async def trade_history():
#     # bsm = BinanceSocketManager(client)

account_trades = await client.get_account()
# traded_symbols = {balance['asset'] + "USDT" for balance in account_trades['balances']}  # Adjust for different pairs
# traded_symbols = {bal['asset']: bal for bal in account_trades['balances'] if float(bal['free'])}
# traded_symbols = [bal for bal in account_trades['balances'] if float(bal['free'])]

# Fetch trades for each symbol
# tasks = [client.get_my_trades(symbol=symbol) for symbol in traded_symbols]
# all_trades = await asyncio.gather(*tasks, return_exceptions=True)
# ic(all_trades[0])

trade_history = [bal for bal in account_trades['balances'] if float(bal['free']) > 0 or float(bal['locked']) > 0]

df = pd.DataFrame(trade_history)  # noqa
df

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7e2d2b7e4890>


,asset,free,locked
0,ETH,0.00001030,0.00000000
1,BNB,0.00000084,0.00000000
2,USDT,0.02679013,0.00000000
3,LRC,0.59400000,0.00000000
4,STX,0.04270000,0.00000000
5,XRP,0.32828900,0.00000000
6,MANA,0.04600000,0.00000000
7,ADA,0.05980000,0.00000000
8,IOST,0.98800000,0.00000000
9,WAN,0.50200000,0.00000000


In [190]:
df = df[(df['free'].astype(float) >= 1) | (df['locked'].astype(float) >= 1)]
# df['asset'].unique()

In [234]:
tasks = [client.get_my_trades(symbol=f'{symbol}USDT') for symbol in df['asset']]
trades = await asyncio.gather(*tasks, return_exceptions=True)
tradesdf = pd.concat([pd.DataFrame(trade) for trade in trades], ignore_index=True)
tradesdf['time'] = pd.to_datetime(tradesdf['time'], unit='ms')
tradesdf = tradesdf.set_index('id').sort_values(by='time').sort_values(by='id')
tradesdf

,symbol,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch
id,,,,,,,,,,,,
2710807,REDUSDT,36848173,-1,0.53980000,172.10000000,92.89958000,0.17210000,RED,2025-03-14 16:21:05.713,True,False,True
2710808,REDUSDT,36848173,-1,0.53980000,66.30000000,35.78874000,0.06630000,RED,2025-03-14 16:21:05.713,True,False,True
2710809,REDUSDT,36848173,-1,0.53980000,13.90000000,7.50322000,0.01390000,RED,2025-03-14 16:21:05.713,True,False,True
2710810,REDUSDT,36848173,-1,0.54010000,4.20000000,2.26842000,0.00420000,RED,2025-03-14 16:21:05.713,True,False,True
2710811,REDUSDT,36848173,-1,0.54010000,21.30000000,11.50413000,0.02130000,RED,2025-03-14 16:21:05.713,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
39390706,ZROUSDT,845588788,-1,3.22300000,103.50000000,333.58050000,0.10350000,ZRO,2025-03-27 06:59:30.675,True,False,True
39390707,ZROUSDT,845588788,-1,3.22300000,99.81000000,321.68763000,0.09981000,ZRO,2025-03-27 06:59:30.675,True,False,True
118591245,EGLDUSDT,1910149777,-1,18.85000000,9.22000000,173.79700000,0.00922000,EGLD,2025-03-27 12:32:15.535,True,False,True


In [237]:
# tradesdf['symbol'].unique()
tradesdf.columns

Index(['symbol', 'orderId', 'orderListId', 'price', 'qty', 'quoteQty',
       'commission', 'commissionAsset', 'time', 'isBuyer', 'isMaker',
       'isBestMatch'],
      dtype='object')

In [ ]:
tradesdf[tradesdf['isBuyer']].sample(10)
tradesdf[(tradesdf['isBuyer']) & (tradesdf['symbol'] == 'BANANAUSDT') & (tradesdf['orderId'] == 383583165)]

In [271]:
comm = 0.001
total = 19.41000000 * 11.87900000
# fee = total * comm
expense = total + 0.01187900
# fee
# total
expense

230.58326899999997

In [238]:
tradesdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 2710807 to 118591247
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   symbol           116 non-null    object        
 1   orderId          116 non-null    int64         
 2   orderListId      116 non-null    int64         
 3   price            116 non-null    object        
 4   qty              116 non-null    object        
 5   quoteQty         116 non-null    object        
 6   commission       116 non-null    object        
 7   commissionAsset  116 non-null    object        
 8   time             116 non-null    datetime64[ns]
 9   isBuyer          116 non-null    bool          
 10  isMaker          116 non-null    bool          
 11  isBestMatch      116 non-null    bool          
dtypes: bool(3), datetime64[ns](1), int64(2), object(6)
memory usage: 9.4+ KB


## Orders

In [ ]:
orders = client.get_all_orders(symbol='BANANAUSDT', limit=10)
orders